## First attempt

In [2]:
import os
from pathlib import Path

import mlflow
import polars as pl
import torch

from typing import Union

import matplotlib.pyplot as plt
import mlflow.pytorch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GINEConv, global_add_pool, global_max_pool, global_mean_pool

BASE_DIR = Path(r"C:\Users\User\Desktop\spark_airflow\chembl\dags\data")
GOLD_DIR = BASE_DIR / "gold"
MLFLOW_TRACKING_URI = "http://localhost:5000"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("GNN_GIN_Experiments")
print(f"MLflow tracking set to: {MLFLOW_TRACKING_URI}")

def get_latest_valid_version(base_path: Path, prefix: str = "version_"):
    if not base_path.exists():
        return None
    
    subdirs = [d for d in base_path.iterdir() if d.is_dir() and d.name.startswith(prefix)]
    subdirs.sort(key=lambda d: d.name, reverse=True)
    
    for subdir in subdirs:
        if (subdir / "scaffold_train.parquet").exists() and (subdir / "scaffold_val.parquet").exists():
            return subdir
            
    return None

LATEST_DATA_PATH = get_latest_valid_version(GOLD_DIR)

if LATEST_DATA_PATH:
    print(f"Ready for training. Data found: {LATEST_DATA_PATH.name}")
else:
    print("Data not found. Ensure the EDA notebook generated the files in the gold directory.")

MLflow tracking set to: http://localhost:5000
Ready for training. Data found: version_v2.0_20260606_1226


In [3]:
def one_hot_encoding(value, choices: list):
    encoding = [0] * (len(choices) + 1)
    index = choices.index(value) if value in choices else -1
    encoding[index] = 1
    return encoding

def smiles_to_graph_advanced(smiles: str, y_val: float, global_features: list):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    node_features = []
    for atom in mol.GetAtoms():
        # NOWE z V1: więcej cech atomu
        features = (
            one_hot_encoding(atom.GetAtomicNum(), [1, 6, 7, 8, 9, 15, 16, 17, 35, 53]) +
            one_hot_encoding(atom.GetDegree(), [0, 1, 2, 3, 4, 5]) +
            one_hot_encoding(atom.GetTotalNumHs(), [0, 1, 2, 3, 4]) +
            one_hot_encoding(int(atom.GetHybridization()), [2, 3, 4, 5]) +
            one_hot_encoding(int(atom.GetChiralTag()), [0, 1, 2, 3]) +
            [1 if atom.GetIsAromatic() else 0,
             1 if atom.IsInRing() else 0,
             atom.GetFormalCharge(),
             atom.GetMass() / 100.0]
        )
        node_features.append(features)

    x = torch.tensor(node_features, dtype=torch.float)
    edges = []
    edge_attrs = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        b_features = one_hot_encoding(
            bond.GetBondType(),
            [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE,
             Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC]
        )
        edges.extend([(i, j), (j, i)])
        edge_attrs.extend([b_features, b_features])

    edge_index = torch.empty((2, 0), dtype=torch.long) if len(edges)==0 else torch.tensor(edges).t().contiguous()
    edge_attr = torch.empty((0, 5), dtype=torch.float) if len(edges)==0 else torch.tensor(edge_attrs, dtype=torch.float)

    y = torch.tensor([[y_val]], dtype=torch.float)
    global_tensor = torch.tensor([global_features], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, global_feats=global_tensor)


class GraphQSARDatasetAdvanced(torch.utils.data.Dataset):
    def __init__(self, parquet_path: Union[str, Path]):
        super().__init__()
        # NOWE z V1: n_measurements jako cecha globalna
        cols = ["canonical_smiles", "pIC50", "aromatic_rings", "hbd", "alogp", "mw_freebase",
                "rtb", "psa", "hba", "qed_weighted", "n_measurements"]
        df = pl.read_parquet(parquet_path).select(cols).drop_nulls()

        self.data_list = []
        for row in df.iter_rows(named=True):
            g_feats = [
                row["aromatic_rings"], row["hbd"], row["alogp"], row["mw_freebase"],
                row["rtb"], row["psa"], row["hba"], row["qed_weighted"],
                row["n_measurements"] / 10.0 # normalizacja
            ]
            data = smiles_to_graph_advanced(row["canonical_smiles"], row["pIC50"], g_feats)
            if data is not None:
                self.data_list.append(data)

    def __len__(self): return len(self.data_list)
    def __getitem__(self, idx): return self.data_list[idx]


class GINAdvanced(nn.Module):
    # Zachowane hiperparametry z V1: hidden_dim=256, num_layers=5, dropout=0.15
    def __init__(self, node_dim: int, edge_dim: int, global_dim: int = 9, hidden_dim: int = 256, num_layers: int = 5, dropout: float = 0.15):
        super().__init__()
        self.dropout = dropout
        self.node_emb = nn.Linear(node_dim, hidden_dim)
        self.edge_emb = nn.Linear(edge_dim, hidden_dim)
        self.virtual_node_emb = nn.Embedding(1, hidden_dim)

        self.convs = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        self.mlp_virtual_nodes = nn.ModuleList() # Dodano brakujące sieci dla wirtualnego węzła
        
        for _ in range(num_layers):
            nn_seq = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim * 2),
                nn.BatchNorm1d(hidden_dim * 2),
                nn.ReLU(),
                nn.Linear(hidden_dim * 2, hidden_dim)
            )
            self.convs.append(GINEConv(nn_seq))
            self.batch_norms.append(nn.BatchNorm1d(hidden_dim))
            
            # Sieci dla aktualizacji wirtualnego węzła z V2, dostosowane wymiarami
            self.mlp_virtual_nodes.append(
                nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim * 2),
                    nn.BatchNorm1d(hidden_dim * 2),
                    nn.ReLU(),
                    nn.Linear(hidden_dim * 2, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU()
                )
            )

        self.global_bn = nn.BatchNorm1d(global_dim)
        self.pool_concat_dim = (hidden_dim * 3) + global_dim

        self.mlp = nn.Sequential(
            nn.Linear(self.pool_concat_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x, edge_index, edge_attr, batch, global_feats):
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)
        batch_size = batch.max().item() + 1
        virtual_node = self.virtual_node_emb(torch.zeros(batch_size, dtype=torch.long, device=x.device))

        # Naprawiona metoda forward, łącząca wirtualne węzły i GINConv
        for i, (conv, bn) in enumerate(zip(self.convs, self.batch_norms)):
            x_res = x
            virtual_node_expanded = virtual_node[batch]
            x = x + virtual_node_expanded
            
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = x + x_res
            
            if i < len(self.convs) - 1:
                virtual_node_pool = global_add_pool(x, batch)
                virtual_node = virtual_node + F.dropout(
                    self.mlp_virtual_nodes[i](virtual_node_pool),
                    p=self.dropout,
                    training=self.training
                )
                
        # Konkatenacja z global_feats (to było pominięte w V1)
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x_add = global_add_pool(x, batch)
        
        global_feats_norm = self.global_bn(global_feats)
        x_pooled = torch.cat([x_mean, x_max, x_add, global_feats_norm], dim=1)
        
        return self.mlp(x_pooled)


def evaluate_gin(model: nn.Module, loader: PyGDataLoader, device: torch.device):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_attr, data.batch, data.global_feats)
            y_true.extend(data.y.cpu().numpy().flatten())
            y_pred.extend(out.cpu().numpy().flatten())
            
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, mae, r2

def train_gin(train_loader: PyGDataLoader, val_loader: PyGDataLoader, node_dim: int, edge_dim: int, global_dim: int, epochs: int = 150, lr: float = 3e-4, run_name: str = "GIN_Run"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Starting computation on: {device}")
    
    # Inicjalizacja modelu z parametrami z V1
    model = GINAdvanced(
        node_dim=node_dim, edge_dim=edge_dim, global_dim=global_dim, 
        hidden_dim=256, num_layers=5, dropout=0.15
    ).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
    criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "epochs": epochs,
            "learning_rate": lr,
            "hidden_dim": 256,
            "num_layers": 5,
            "model_architecture": "GIN_Advanced_v1_Fixed",
        })
        
        for epoch in range(epochs):
            model.train()
            train_loss = 0.0
            for data in train_loader:
                data = data.to(device)
                optimizer.zero_grad()
                out = model(data.x, data.edge_index, data.edge_attr, data.batch, data.global_feats)
                loss = criterion(out, data.y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * data.num_graphs
                
            avg_train = train_loss / len(train_loader.dataset)
            val_mse, val_mae, val_r2 = evaluate_gin(model, val_loader, device)
            scheduler.step(val_mse)
            
            train_losses.append(avg_train)
            val_losses.append(val_mse)
            
            mlflow.log_metric("train_mse", avg_train, step=epoch)
            mlflow.log_metric("val_mse", val_mse, step=epoch)
            mlflow.log_metric("val_mae", val_mae, step=epoch)
            mlflow.log_metric("val_r2", val_r2, step=epoch)
            
            if epoch % 5 == 0 or epoch == epochs - 1:
                print(f"Epoch {epoch:03d}: Train MSE {avg_train:.4f} | Val MSE {val_mse:.4f} | Val R2 {val_r2:.4f}")

        model.cpu()
        try:
            mlflow.pytorch.log_model(model, "model")
        except Exception as e:
            print(f"MLflow model logging failed: {e}, saving state_dict only")
            mlflow.log_artifact("best_model.pt")
        
        plt.figure(figsize=(10, 6))
        plt.plot(train_losses, label='Train MSE', color='#4C72B0', linewidth=2)
        plt.plot(val_losses, label='Val MSE', color='#DD8452', linewidth=2)
        plt.title(f'Learning Curve: {run_name}', fontsize=14)
        plt.xlabel('Epochs', fontsize=12)
        plt.ylabel('Mean Squared Error', fontsize=12)
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plot_filename = f"learning_curve_{run_name.replace(' ', '_')}.png"
        plt.savefig(plot_filename)
        plt.close()
        
        mlflow.log_artifact(plot_filename)
        mlflow.pytorch.log_model(model, "model")
        
        if os.path.exists(plot_filename):
            os.remove(plot_filename)
            
    return model, val_losses[-1], val_mae, val_r2


In [5]:
print(f"Loading data from: {LATEST_DATA_PATH.name}...")

train_scaffold_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_train.parquet")
val_scaffold_ds = GraphQSARDatasetAdvanced(LATEST_DATA_PATH / "scaffold_val.parquet")

train_scaffold_loader = PyGDataLoader(
    train_scaffold_ds, batch_size=64, shuffle=True, num_workers=0, pin_memory=True
)
val_scaffold_loader = PyGDataLoader(
    val_scaffold_ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True
)

sample_batch = next(iter(train_scaffold_loader))
n_dim = sample_batch.num_node_features
e_dim = sample_batch.num_edge_features
g_dim = sample_batch.global_feats.shape[1]

print(f"Detected features -> Node: {n_dim}, Edge: {e_dim}, Global: {g_dim}")
print("Starting GIN Advanced training...")

model_scaff, mse_scaff, mae_scaff, r2_scaff = train_gin(
    train_scaffold_loader, val_scaffold_loader,
    node_dim=n_dim, edge_dim=e_dim, global_dim=g_dim,
    epochs=5, lr=3e-4, run_name="GIN_v2_clean"
)

print(f"\nFINAL RESULT - MSE: {mse_scaff:.4f}, MAE: {mae_scaff:.4f}, R2: {r2_scaff:.4f}")

Loading data from: version_v2.0_20260606_1226...
Detected features -> Node: 38, Edge: 5, Global: 9
Starting GIN Advanced training...
Starting computation on: cpu


2026/06/06 14:11:29 INFO mlflow.tracking.fluent: Experiment with name 'GNN_GIN_Experiments' does not exist. Creating a new experiment.


Epoch 000: Train MSE 29.7446 | Val MSE 22.9114 | Val R2 -14.8861
Epoch 004: Train MSE 1.7794 | Val MSE 7.4680 | Val R2 -4.1781


2026/06/06 14:17:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/06/06 14:17:32 INFO mlflow.tracking._tracking_service.client: 🏃 View run GIN_v2_clean at: http://localhost:5000/#/experiments/1/runs/c86672ccab034785a267462e25884f2d.
2026/06/06 14:17:32 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/1.



FINAL RESULT - MSE: 7.4680, MAE: 1.1669, R2: -4.1781


## Chempion model

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
import mlflow
import torch

os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'

BASE_DIR = Path('/content/drive/MyDrive/chembl_project/data')
GOLD_DIR = BASE_DIR / 'gold'
MLFLOW_DIR = Path('/content/drive/MyDrive/chembl_project/mlruns')
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_TRACKING_URI = f'file://{MLFLOW_DIR}'

os.environ['MLFLOW_TRACKING_URI'] = MLFLOW_TRACKING_URI
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("GNN_GIN_Experiments")

print(f'✅ MLflow: {MLFLOW_TRACKING_URI}')
print(f'✅ GPU: {torch.cuda.is_available()} - {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
torch.backends.cudnn.benchmark = True

def get_all_valid_versions(base_path: Path):
    # Fetch all directories matching the specific prefix
    if not base_path.exists(): 
        return []
    
    all_dirs = []
    for subdir in base_path.iterdir():
        if subdir.is_dir() and subdir.name.startswith("version_"):
            all_dirs.append(subdir)
            
    return sorted(all_dirs, key=lambda x: x.name)

ALL_DATA_PATHS = get_all_valid_versions(GOLD_DIR)

print(f"📂 Found {len(ALL_DATA_PATHS)} datasets:")
for path in ALL_DATA_PATHS:
    # Print file count inside each directory to verify structure
    files_inside = [f.name for f in path.iterdir() if f.is_file()]
    print(f"  - {path.name} | Files inside: {len(files_inside)}")

ModuleNotFoundError: No module named 'google.colab'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ MLflow: file:///content/drive/MyDrive/chembl_project/mlruns
✅ GPU: True - Tesla T4
📂 Found 4 datasets:
  - version_v2.0_20260606_1021 | Files inside: 7
  - version_v2.0_20260606_1226 | Files inside: 7
  - version_v3.0_205_20260606_1447 | Files inside: 7
  - version_v3.0_205_20260606_1520 | Files inside: 7

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import polars as pl, mlflow, mlflow.pytorch, json, datetime
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GINEConv, global_mean_pool, global_max_pool, global_add_pool
from rdkit import Chem
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def one_hot_encoding(value, choices):
    enc = [0]*(len(choices)+1); idx = choices.index(value) if value in choices else -1; enc[idx]=1; return enc

def smiles_to_graph_advanced(smiles, y_val, g_feats):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None
    x = torch.tensor([one_hot_encoding(a.GetAtomicNum(),[1,6,7,8,9,15,16,17,35,53])+
                      one_hot_encoding(a.GetDegree(),[0,1,2,3,4,5])+
                      one_hot_encoding(a.GetTotalNumHs(),[0,1,2,3,4])+
                      one_hot_encoding(int(a.GetHybridization()),[2,3,4,5])+
                      one_hot_encoding(int(a.GetChiralTag()),[0,1,2,3])+
                      [a.GetIsAromatic(), a.IsInRing(), a.GetFormalCharge(), a.GetMass()/100]
                      for a in mol.GetAtoms()], dtype=torch.float)
    edges, attrs = [], []
    for b in mol.GetBonds():
        i,j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        f = one_hot_encoding(b.GetBondType(),[Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE, Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC])
        edges+= [(i,j),(j,i)]; attrs+=[f,f]
    edge_index = torch.tensor(edges).t().contiguous() if edges else torch.empty((2,0),dtype=torch.long)
    edge_attr = torch.tensor(attrs, dtype=torch.float) if attrs else torch.empty((0,5))
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=torch.tensor([[y_val]]), global_feats=torch.tensor([g_feats], dtype=torch.float))

class GraphQSARDatasetAdvanced(torch.utils.data.Dataset):
    def __init__(self, parquet_path, augment=False):
        df = pl.read_parquet(parquet_path)
        has_mw = "mw_freebase" in df.columns
        
        # POPRAWKA: Usunięto "n_measurements" z listy kolumn
        cols = ["canonical_smiles","pIC50","aromatic_rings","hbd","alogp","rtb","psa","hba","qed_weighted","ecfp_2048"]
        cols += ["mw_freebase"] if has_mw else ["heavy_atoms"]
        df = df.select(cols).drop_nulls()
        
        self.data_list = []
        for r in df.iter_rows(named=True):
            mw = r["mw_freebase"] if has_mw else r["heavy_atoms"]*12
            ecfp = list(r["ecfp_2048"])[:64]  # 64 bity z Twojego parquet
            
            # POPRAWKA: Usunięto r["n_measurements"]/10. Baza g_base ma teraz 8 elementów.
            g_base = [r["aromatic_rings"],r["hbd"],r["alogp"],mw/500,r["rtb"],r["psa"]/100,r["hba"],r["qed_weighted"]]
            g_feats = g_base + ecfp  # teraz global_dim = 72
            
            smis = [r["canonical_smiles"]]
            if augment:
                m = Chem.MolFromSmiles(r["canonical_smiles"])
                if m:
                    for _ in range(2):
                        smis.append(Chem.MolToSmiles(m, doRandom=True))
            for smi in smis:
                d = smiles_to_graph_advanced(smi, r["pIC50"], g_feats)
                if d: self.data_list.append(d)
                
    def __len__(self): return len(self.data_list)
    def __getitem__(self, i): return self.data_list[i]

class GINAdvanced(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dim=128, num_layers=3, dropout=0.2):
        super().__init__()
        self.node_emb = nn.Linear(node_dim, hidden_dim)
        self.edge_emb = nn.Linear(edge_dim, hidden_dim)
        self.convs = nn.ModuleList(); self.bns = nn.ModuleList()
        for _ in range(num_layers):
            seq = nn.Sequential(nn.Linear(hidden_dim, hidden_dim*2), nn.BatchNorm1d(hidden_dim*2), nn.ReLU(), nn.Linear(hidden_dim*2, hidden_dim))
            self.convs.append(GINEConv(seq)); self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.gbn = nn.BatchNorm1d(global_dim)
        self.mlp = nn.Sequential(nn.Linear(hidden_dim*3+global_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(hidden_dim, hidden_dim//2), nn.BatchNorm1d(hidden_dim//2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim//2,1))
    def forward(self,x,ei,ea,batch,gf):
        x = self.node_emb(x); ea = self.edge_emb(ea)
        for conv,bn in zip(self.convs,self.bns):
            x = F.relu(bn(conv(x,ei,ea) + x))
        xm = global_mean_pool(x,batch); xa = global_max_pool(x,batch); xs = global_add_pool(x,batch)
        return self.mlp(torch.cat([xm,xa,xs,self.gbn(gf)],1))

def evaluate_gin(model, loader, device):
    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for d in loader:
            d = d.to(device)
            out = model(d.x, d.edge_index, d.edge_attr, d.batch, d.global_feats)
            yt.extend(d.y.cpu().numpy().flatten()); yp.extend(out.cpu().numpy().flatten())
    return mean_squared_error(yt,yp), mean_absolute_error(yt,yp), r2_score(yt,yp)

def train_gin(train_loader,val_loader,nd,ed,gd,epochs=100,lr=3e-4,run_name="GIN_Run"):
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model=GINAdvanced(nd,ed,gd).to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=5e-4)
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max',factor=0.5,patience=8)
    crit=nn.SmoothL1Loss()
    hparams={"hidden":128,"layers":3,"dropout":0.2,"lr":lr,"ecfp":True,"augment":True,"global_dim":gd,"params":sum(p.numel() for p in model.parameters())}
    
    train_hist, val_hist = [], []
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(hparams); best=-float('inf'); patience=0
        torch.save(model.state_dict(),"/content/best.pt")
        for ep in range(epochs):
            model.train(); tl=0
            for d in train_loader:
                d=d.to(device); opt.zero_grad()
                loss = crit(model(d.x,d.edge_index,d.edge_attr,d.batch,d.global_feats), d.y)
                loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
                tl+=loss.item()*d.num_graphs
            at=tl/len(train_loader.dataset)
            vm,va,vr = evaluate_gin(model,val_loader,device); sched.step(vr)
            train_hist.append(at); val_hist.append(vm)
            mlflow.log_metrics({"train":at,"val_r2":vr,"val_mse":vm},step=ep)
            if ep % 5 == 0 or ep == epochs - 1:
                print(f"Ep{ep:03d} train loss {at:.3f} val_r2 {vr:.4f} best_r2 {best:.4f}")
            if vr>best: best=vr; patience=0; torch.save(model.state_dict(),"/content/best.pt")
            else: patience+=1
            if patience>=15: print("Early stopping triggered."); break
            
        model.load_state_dict(torch.load("/content/best.pt"))
        plt.figure(figsize=(8,4)); plt.plot(train_hist,label='train_loss'); plt.plot(val_hist,label='val_mse'); plt.legend(); plt.grid(alpha=0.3)
        plot_path = Path(f"/content/{run_name}_curve.png"); plt.savefig(plot_path,dpi=150); plt.close(); mlflow.log_artifact(str(plot_path))
        mlflow.pytorch.log_model(model.cpu(), "model"); model = model.to(device)
        
    return model, vm, va, best, hparams

In [ ]:
if not ALL_DATA_PATHS:
    print("⚠️ Nie wykryto żadnych zbiorów danych w folderze GOLD.")
else:
    all_results = {}

    # Pętla przez każdy z wykrytych zbiorów danych
    for idx, data_path in enumerate(ALL_DATA_PATHS):
        dataset_name = data_path.name
        run_name = f"GIN_Scaffold_{dataset_name}"
        
        print(f"\n{'='*60}")
        print(f"🚀 [{idx+1}/{len(ALL_DATA_PATHS)}] Rozpoczynam trening na zbiorze: {dataset_name}")
        print(f"{'='*60}")
        
        # Inicjalizacja datasetów dla aktualnej iteracji
        train_ds = GraphQSARDatasetAdvanced(data_path / "scaffold_train.parquet", augment=True)
        val_ds = GraphQSARDatasetAdvanced(data_path / "scaffold_val.parquet", augment=False)

        train_loader = PyGDataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
        val_loader = PyGDataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

        s = next(iter(train_loader))
        print(f"📊 Parametry z {dataset_name}: Node={s.num_node_features} Edge={s.num_edge_features} Global={s.global_feats.shape[1]} Train_Size={len(train_ds)}")

        # Trenowanie i logowanie modelu pod nową nazwą eksperymentu
        model, mse, mae, val_r2, hparams = train_gin(
            train_loader, val_loader, 
            s.num_node_features, s.num_edge_features, s.global_feats.shape[1],
            epochs=100, lr=3e-4, run_name=run_name
        )

        print(f"\n🏆 WYNIK WALIDACJI ({dataset_name}): R2 = {val_r2:.4f}")

        # Walidacja testowa (jeśli plik istnieje w danym zbiorze)
        test_path = data_path / "scaffold_test.parquet"
        test_r2 = None
        if test_path.exists():
            test_ds = GraphQSARDatasetAdvanced(test_path, augment=False)
            test_loader = PyGDataLoader(test_ds, batch_size=32, shuffle=False)
            tm, ta, test_r2 = evaluate_gin(model, test_loader, torch.device('cuda'))
            print(f"🧪 WYNIK TESTOWY ({dataset_name}): R2 = {test_r2:.4f}")

        # Zapisujemy wyniki z tej iteracji
        all_results[dataset_name] = {
            "val_r2": float(val_r2),
            "test_r2": float(test_r2) if test_r2 is not None else "BRAK",
            "hyperparameters": hparams
        }

    # --- PODSUMOWANIE ---
    print(f"\n{'*'*60}")
    print("📋 PODSUMOWANIE WYNIKÓW DLA WSZYSTKICH ZBIORÓW DANYCH:")
    print(f"{'*'*60}")
    
    for ds_name, res in all_results.items():
        test_str = f"{res['test_r2']:.4f}" if isinstance(res['test_r2'], float) else res['test_r2']
        print(f"▶ Zbiór: {ds_name} | VAL R2: {res['val_r2']:.4f} | TEST R2: {test_str}")

    # Zrzut całego podsumowania do pliku na dysku
    summary_path = BASE_DIR / "multi_dataset_results_summary.json"
    with open(summary_path, "w") as f:
        json.dump(all_results, f, indent=4)
        
    print(f"\n✅ Zapisano zbiorczy plik wyników do: {summary_path.name}")

🏆 WYNIK WALIDACJI (version_v2.0_20260606_1226): R2 = 0.4640
🧪 WYNIK TESTOWY (version_v2.0_20260606_1226): R2 = 0.505
🏆 WYNIK WALIDACJI (version_v3.0_205_20260606_1520): R2 = 0.5278
🧪 WYNIK TESTOWY (version_v3.0_205_20260606_1520): R2 = -0.2353

************************************************************
📋 PODSUMOWANIE WYNIKÓW DLA WSZYSTKICH ZBIORÓW DANYCH:
************************************************************
▶ Zbiór: version_v2.0_20260606_1226 | VAL R2: 0.4640 | TEST R2: 0.5059
▶ Zbiór: version_v3.0_205_20260606_1520 | VAL R2: 0.5278 | TEST R2: mary.json9

In [ ]:
import json

# Lista zbiorów, które już zostały przetrenowane i chcemy je pominąć
ALREADY_PROCESSED = [
    "version_v2.0_20260606_1226",
    "version_v3.0_205_20260606_1520"
]

# Wyfiltrowanie tylko tych ścieżek, których jeszcze nie liczyliśmy
paths_to_run = [p for p in ALL_DATA_PATHS if p.name not in ALREADY_PROCESSED]

if not paths_to_run:
    print("⚠️ Wszystkie zbiory z folderu GOLD zostały już przetworzone!")
else:
    # 1. Wczytanie starych wyników, jeśli plik istnieje (aby ich nie nadpisać, lecz dopisać nowe)
    summary_path = BASE_DIR / "multi_dataset_results_summary.json"
    if summary_path.exists():
        with open(summary_path, "r") as f:
            all_results = json.load(f)
        print(f"🔄 Wczytano {len(all_results)} zapisanych wcześniej wyników z pliku podsumowania.")
    else:
        all_results = {}

    # 2. Pętla tylko przez POZOSTAŁE zbiory danych
    for idx, data_path in enumerate(paths_to_run):
        dataset_name = data_path.name
        run_name = f"GIN_Scaffold_{dataset_name}"
        
        print(f"\n{'='*60}")
        print(f"🚀 [{idx+1}/{len(paths_to_run)}] Rozpoczynam trening na zbiorze: {dataset_name}")
        print(f"{'='*60}")
        
        # Inicjalizacja datasetów dla aktualnej iteracji
        train_ds = GraphQSARDatasetAdvanced(data_path / "scaffold_train.parquet", augment=True)
        val_ds = GraphQSARDatasetAdvanced(data_path / "scaffold_val.parquet", augment=False)

        train_loader = PyGDataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
        val_loader = PyGDataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

        s = next(iter(train_loader))
        print(f"📊 Parametry z {dataset_name}: Node={s.num_node_features} Edge={s.num_edge_features} Global={s.global_feats.shape[1]} Train_Size={len(train_ds)}")

        # Trenowanie i logowanie modelu pod nową nazwą eksperymentu
        model, mse, mae, val_r2, hparams = train_gin(
            train_loader, val_loader, 
            s.num_node_features, s.num_edge_features, s.global_feats.shape[1],
            epochs=100, lr=3e-4, run_name=run_name
        )

        print(f"\n🏆 WYNIK WALIDACJI ({dataset_name}): R2 = {val_r2:.4f}")

        # Walidacja testowa (jeśli plik istnieje w danym zbiorze)
        test_path = data_path / "scaffold_test.parquet"
        test_r2 = None
        if test_path.exists():
            test_ds = GraphQSARDatasetAdvanced(test_path, augment=False)
            test_loader = PyGDataLoader(test_ds, batch_size=32, shuffle=False)
            tm, ta, test_r2 = evaluate_gin(model, test_loader, torch.device('cuda'))
            print(f"🧪 WYNIK TESTOWY ({dataset_name}): R2 = {test_r2:.4f}")

        # Zapisujemy wyniki z tej iteracji do zbiorczego słownika
        all_results[dataset_name] = {
            "val_r2": float(val_r2),
            "test_r2": float(test_r2) if test_r2 is not None else "BRAK",
            "hyperparameters": hparams
        }

    # --- PODSUMOWANIE ---
    print(f"\n{'*'*60}")
    print("📋 PEŁNE PODSUMOWANIE WYNIKÓW DLA WSZYSTKICH ZBIORÓW DANYCH:")
    print(f"{'*'*60}")
    
    for ds_name, res in all_results.items():
        test_str = f"{res['test_r2']:.4f}" if isinstance(res['test_r2'], float) else res['test_r2']
        print(f"▶ Zbiór: {ds_name} | VAL R2: {res['val_r2']:.4f} | TEST R2: {test_str}")

    # Zrzut całego podsumowania (stare + nowe wyniki) do pliku na dysku
    with open(summary_path, "w") as f:
        json.dump(all_results, f, indent=4)
        
    print(f"\n✅ Zaktualizowano i zapisano zbiorczy plik wyników do: {summary_path.name}")

🏆 WYNIK WALIDACJI (version_v2.0_20260606_1021): R2 = 0.3853
🧪 WYNIK TESTOWY (version_v2.0_20260606_1021): R2 = 0.383
🏆 WYNIK WALIDACJI (version_v3.0_205_20260606_1447): R2 = -0.5438
🧪 WYNIK TESTOWY (version_v3.0_205_20260606_1447): R2 = 0.5239

************************************************************
📋 PEŁNE PODSUMOWANIE WYNIKÓW DLA WSZYSTKICH ZBIORÓW DANYCH:
************************************************************
▶ Zbiór: version_v2.0_20260606_1226 | VAL R2: 0.4640 | TEST R2: 0.5059
▶ Zbiór: version_v3.0_205_20260606_1520 | VAL R2: 0.5278 | TEST R2: -0.2353
▶ Zbiór: version_v2.0_20260606_1021 | VAL R2: 0.3853 | TEST R2: 0.3832
▶ Zbiór: version_v3.0_205_20260606_1447 | VAL R2: -0.5438 | TEST R2: 0.52392

In [ ]:
import gc
import torch
import optuna
import mlflow
import torch.nn as nn
from torch_geometric.loader import DataLoader as PyGDataLoader

v2_path = None
for p in ALL_DATA_PATHS:
    if p.name == "version_v2.0_20260606_1226":
        v2_path = p
        break

if v2_path is None:
    raise ValueError("Target dataset v2.0_20260606_1226 not found.")

train_ds = GraphQSARDatasetAdvanced(v2_path / "scaffold_train.parquet", augment=True)
val_ds = GraphQSARDatasetAdvanced(v2_path / "scaffold_val.parquet", augment=False)

train_loader = PyGDataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
val_loader = PyGDataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

s = next(iter(train_loader))
n_dim, e_dim, g_dim = s.num_node_features, s.num_edge_features, s.global_feats.shape[1]

def objective(trial):
    hidden_dim = trial.suggest_categorical("hidden_dim", [128, 256, 512])
    num_layers = trial.suggest_int("num_layers", 3, 5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4, step=0.1)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = GINAdvanced(
        node_dim=n_dim, edge_dim=e_dim, global_dim=g_dim, 
        hidden_dim=hidden_dim, num_layers=num_layers, dropout=dropout
    ).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    crit = nn.SmoothL1Loss() 
    
    epochs = 30 
    best_val_r2 = -float("inf")
    run_name = f"Optuna_v2_1226_{trial.number}"
    
    with mlflow.start_run(run_name=run_name, nested=True):
        mlflow.log_params(trial.params)
        
        for ep in range(epochs):
            model.train()
            for d in train_loader:
                d = d.to(device)
                optimizer.zero_grad()
                out = model(d.x, d.edge_index, d.edge_attr, d.batch, d.global_feats)
                loss = crit(out, d.y)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
            vm, va, vr = evaluate_gin(model, val_loader, device)
            scheduler.step(vr)
            
            if vr > best_val_r2:
                best_val_r2 = vr
                
            mlflow.log_metric("val_r2", vr, step=ep)
            trial.report(vr, ep)
            
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()
                
    del model, optimizer, crit, scheduler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return best_val_r2

mlflow.set_experiment("GNN_Optuna_V2_Gold")

study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10))
study.optimize(objective, n_trials=30)

print("Optimization finished.")
for key, value in study.best_params.items():
    print(f"{key}: {value}")
print(f"Best R2: {study.best_value:.4f}")

[I 2026-06-06 15:47:04,138] Trial 28 pruned. 
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
[I 2026-06-06 15:48:15,801] Trial 29 pruned. 
Optimization finished.
hidden_dim: 512
num_layers: 4
dropout: 0.2
lr: 0.00017467646860663965
weight_decay: 4.41468128263608e-06
Best R2: 0.5038

In [ ]:
import matplotlib.pyplot as plt
import json, pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch.nn as nn

# 1. NAJLEPSZE PARAMETRY Z OPTUNY
best = study.best_params
print("Trenuję finalny model na:", best)

device = torch.device("cuda")
# załaduj dane
train_ds = GraphQSARDatasetAdvanced(v2_path / "scaffold_train.parquet", augment=True)
val_ds = GraphQSARDatasetAdvanced(v2_path / "scaffold_val.parquet", augment=False)
test_ds = GraphQSARDatasetAdvanced(v2_path / "scaffold_test.parquet", augment=False)

train_loader = PyGDataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
val_loader = PyGDataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2)
test_loader = PyGDataLoader(test_ds, batch_size=256, shuffle=False)

# wymiary
s = next(iter(train_loader))
n_dim, e_dim, g_dim = s.num_node_features, s.edge_attr.shape[1], s.global_feats.shape[1]

# 2. MODEL - FAZA 1
model = GINAdvanced(
    node_dim=n_dim, edge_dim=e_dim, global_dim=g_dim,
    hidden_dim=best["hidden_dim"],
    num_layers=best["num_layers"],
    dropout=best["dropout"]
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=best["lr"], weight_decay=best["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
crit = nn.SmoothL1Loss()

mlflow.set_experiment("GNN_Final_V2_Best")
train_hist, val_r2_hist, val_mse_hist = [], [], []
best_val = -1

with mlflow.start_run(run_name="Final_GIN_v2_best_long"):
    mlflow.log_params(best)
    mlflow.log_param("dataset", v2_path.name)
    mlflow.log_param("phase1_epochs", 100)

    # FAZA 1 - 100 epok zamiast 60
    for ep in range(100):
        model.train(); tl=0
        for d in train_loader:
            d=d.to(device); optimizer.zero_grad()
            loss = crit(model(d.x,d.edge_index,d.edge_attr,d.batch,d.global_feats), d.y)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
            tl += loss.item()*d.num_graphs
        train_loss = tl/len(train_loader.dataset)

        vm, va, vr = evaluate_gin(model, val_loader, device)
        scheduler.step(vr)

        train_hist.append(train_loss); val_r2_hist.append(vr); val_mse_hist.append(vm)
        mlflow.log_metrics({"train_loss":train_loss,"val_r2":vr,"val_mse":vm}, step=ep)

        if vr > best_val:
            best_val = vr
            torch.save(model.state_dict(), "/content/best_final.pt")
        if ep % 10 == 0:
            print(f"Ep{ep:03d} loss {train_loss:.3f} val_R2 {vr:.4f}")

    # 3. FAZA 2 - FINE TUNING (moja magia)
    print("=== FINE-TUNING ===")
    model_ft = GINAdvanced(
        node_dim=n_dim, edge_dim=e_dim, global_dim=g_dim,
        hidden_dim=best["hidden_dim"],
        num_layers=best["num_layers"],
        dropout=0.1 # mniejszy dropout
    ).to(device)
    model_ft.load_state_dict(torch.load("/content/best_final.pt"), strict=False)

    opt_ft = torch.optim.AdamW(model_ft.parameters(), lr=best["lr"]*0.3, weight_decay=best["weight_decay"])
    sched_ft = torch.optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=40)

    for ep in range(40):
        model_ft.train(); tl=0
        for d in train_loader:
            d=d.to(device); opt_ft.zero_grad()
            loss = crit(model_ft(d.x,d.edge_index,d.edge_attr,d.batch,d.global_feats), d.y)
            loss.backward(); opt_ft.step()
            tl += loss.item()*d.num_graphs
        train_loss = tl/len(train_loader.dataset)
        vm, va, vr = evaluate_gin(model_ft, val_loader, device)
        sched_ft.step()

        train_hist.append(train_loss); val_r2_hist.append(vr); val_mse_hist.append(vm)
        mlflow.log_metrics({"ft_train_loss":train_loss,"ft_val_r2":vr,"ft_val_mse":vm}, step=100+ep)

        if vr > best_val:
            best_val = vr
            torch.save(model_ft.state_dict(), "/content/best_final_ft.pt")
        print(f"FT Ep{ep:02d} loss {train_loss:.3f} val_R2 {vr:.4f}")

    # 4. LEARNING CURVE
    plt.figure(figsize=(9,4))
    plt.plot(train_hist, label='train_loss'); plt.plot(val_mse_hist, label='val_mse')
    plt.axvline(100, color='red', linestyle='--', label='fine-tune start')
    plt.legend(); plt.grid(alpha=0.3); plt.title("Final Learning Curve (100+40 epok)")
    plt.savefig("/content/final_learning_curve.png", dpi=150); plt.close()
    mlflow.log_artifact("/content/final_learning_curve.png")

    # 5. ŁADUJ BEST I TESTUJ
    model_ft.load_state_dict(torch.load("/content/best_final_ft.pt"))
    model_ft.eval()

    def get_preds(loader, mdl):
        yt, yp = [], []
        with torch.no_grad():
            for d in loader:
                d=d.to(device)
                out = mdl(d.x,d.edge_index,d.edge_attr,d.batch,d.global_feats)
                yt.extend(d.y.cpu().numpy().flatten()); yp.extend(out.cpu().numpy().flatten())
        return yt, yp

    yv, pv = get_preds(val_loader, model_ft)
    yt, pt = get_preds(test_loader, model_ft)

    val_metrics = {"R2":r2_score(yv,pv),"MSE":mean_squared_error(yv,pv),"MAE":mean_absolute_error(yv,pv)}
    test_metrics = {"R2":r2_score(yt,pt),"MSE":mean_squared_error(yt,pt),"MAE":mean_absolute_error(yt,pt)}

    # 6. PARITY PLOT
    fig, ax = plt.subplots(1,2,figsize=(10,4))
    ax[0].scatter(yv,pv,alpha=0.5,s=10); ax[0].plot([3,10],[3,10],'r--'); ax[0].set_title(f"VAL R2={val_metrics['R2']:.3f}")
    ax[1].scatter(yt,pt,alpha=0.5,s=10); ax[1].plot([3,10],[3,10],'r--'); ax[1].set_title(f"TEST R2={test_metrics['R2']:.3f}")
    for a in ax: a.set_xlabel("true pIC50"); a.set_ylabel("pred pIC50"); a.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig("/content/final_parity.png", dpi=150); plt.close()
    mlflow.log_artifact("/content/final_parity.png")

    # 7. ZAPISZ WSZYSTKO
    report = {
        "model": "GINAdvanced",
        "dataset": v2_path.name,
        "best_params": best,
        "training": {"phase1_epochs":100, "phase2_finetune_epochs":40, "dropout_ft":0.1, "lr_ft": best["lr"]*0.3},
        "train_n": len(train_ds), "val_n": len(val_ds), "test_n": len(test_ds),
        "val_metrics": val_metrics,
        "test_metrics": test_metrics
    }
    with open("/content/final_best_model_report.json","w") as f: json.dump(report,f,indent=2)
    pd.DataFrame({"y_true":yt,"y_pred":pt}).to_csv("/content/final_test_predictions.csv", index=False)

    mlflow.log_artifact("/content/final_best_model_report.json")
    mlflow.pytorch.log_model(model_ft.cpu(), "final_model_ft")

print("Zapisano:")
print("- /content/final_learning_curve.png")
print("- /content/final_parity.png")
print("- /content/final_best_model_report.json")
print("VAL:", val_metrics)
print("TEST:", test_metrics)

FT Ep39 loss 0.036 val_R2 0.5136

## Klasyfikacja dla testu

In [3]:
import glob
import pandas as pd

folder_path = r"C:\Users\User\Desktop\spark_airflow\chembl\dags\data\gold\version_v2.0_20260606_1226"

parquet_files = glob.glob(f"{folder_path}/*.parquet")

if parquet_files:
    df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
    pd.set_option('display.max_columns', None)
    print(f"Pomyślnie wczytano {len(parquet_files)} plików Parquet.")
    print(df.head())
else:
    print(f"Nie znaleziono plików .parquet w katalogu: {folder_path}")

Pomyślnie wczytano 6 plików Parquet.
  molecule_chembl_id     pIC50  pIC50_std  n_measurements  mw_freebase  hba  \
0      CHEMBL2087360  8.022879   0.956137               2       419.84  7.0   
1      CHEMBL1172803  5.441291   0.257332               1       335.75  6.0   
2      CHEMBL5767790  6.759637   0.707107               2       407.41  7.0   
3      CHEMBL3947406  8.397940   0.485068               1       345.40  5.0   
4        CHEMBL94431  8.420216   0.480557               1       299.31  5.0   

                                   canonical_smiles  aromatic_rings  rtb  \
0        Fc1ccc(Nc2ncnc3cc4c(cc23)OCCOCCOCCO4)cc1Cl             3.0  2.0   
1   Cc1ncc([N+](=O)[O-])n1CCOC(=O)/C=C/c1ccc(Cl)cc1             2.0  6.0   
2  C=CC(=O)Nc1cccc(Nc2nc(Nc3ccc4c(c3)OCCO4)ncc2F)c1             3.0  6.0   
3       COc1ccccc1-c1cc2c(N[C@H](C)c3ccccc3)ncnc2o1             4.0  5.0   
4                  COc1cc2ncnc(Nc3cccc(F)c3)c2cc1OC             3.0  4.0   

   qed_weighted  hbd    psa  al

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
import mlflow
import mlflow.pytorch
import json
import gc
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GINEConv, global_mean_pool, global_max_pool, global_add_pool
from rdkit import Chem
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, roc_curve

def one_hot_encoding(value, choices):
    enc = [0] * (len(choices) + 1)
    idx = choices.index(value) if value in choices else -1
    enc[idx] = 1
    return enc

def smiles_to_graph_advanced(smiles, y_val, g_feats):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: 
        return None
    
    x = torch.tensor([one_hot_encoding(a.GetAtomicNum(), [1,6,7,8,9,15,16,17,35,53]) +
                      one_hot_encoding(a.GetDegree(), [0,1,2,3,4,5]) +
                      one_hot_encoding(a.GetTotalNumHs(), [0,1,2,3,4]) +
                      one_hot_encoding(int(a.GetHybridization()), [2,3,4,5]) +
                      one_hot_encoding(int(a.GetChiralTag()), [0,1,2,3]) +
                      [a.GetIsAromatic(), a.IsInRing(), a.GetFormalCharge(), a.GetMass() / 100]
                      for a in mol.GetAtoms()], dtype=torch.float)
    
    edges, attrs = [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        f = one_hot_encoding(b.GetBondType(), [Chem.rdchem.BondType.SINGLE, Chem.rdchem.BondType.DOUBLE, Chem.rdchem.BondType.TRIPLE, Chem.rdchem.BondType.AROMATIC])
        edges += [(i, j), (j, i)]
        attrs += [f, f]
        
    edge_index = torch.tensor(edges).t().contiguous() if edges else torch.empty((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(attrs, dtype=torch.float) if attrs else torch.empty((0, 5))
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, 
                y=torch.tensor([[y_val]], dtype=torch.float), 
                global_feats=torch.tensor([g_feats], dtype=torch.float))

class GraphQSARDatasetClassification(torch.utils.data.Dataset):
    def __init__(self, parquet_path, augment=False, activity_threshold=6.0):
        df = pl.read_parquet(parquet_path)
        has_mw = "mw_freebase" in df.columns
        cols = ["canonical_smiles", "pIC50", "aromatic_rings", "hbd", "alogp", "rtb", "psa", "hba", "qed_weighted", "ecfp_2048"]
        cols += ["mw_freebase"] if has_mw else ["heavy_atoms"]
        df = df.select(cols).drop_nulls()
        
        self.data_list = []
        for r in df.iter_rows(named=True):
            mw = r["mw_freebase"] if has_mw else r["heavy_atoms"] * 12
            ecfp = list(r["ecfp_2048"])[:64]
            g_base = [r["aromatic_rings"], r["hbd"], r["alogp"], mw / 500, r["rtb"], r["psa"] / 100, r["hba"], r["qed_weighted"]]
            g_feats = g_base + ecfp
            
            y_val = 1.0 if r["pIC50"] >= activity_threshold else 0.0
            smis = [r["canonical_smiles"]]
            
            if augment:
                m = Chem.MolFromSmiles(r["canonical_smiles"])
                if m:
                    for _ in range(2):
                        smis.append(Chem.MolToSmiles(m, doRandom=True))
                        
            for smi in smis:
                d = smiles_to_graph_advanced(smi, y_val, g_feats)
                if d: 
                    self.data_list.append(d)
                
    def __len__(self): 
        return len(self.data_list)
        
    def __getitem__(self, i): 
        return self.data_list[i]

class GINAdvancedClassifier(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dim=128, num_layers=3, dropout=0.2):
        super().__init__()
        self.node_emb = nn.Linear(node_dim, hidden_dim)
        self.edge_emb = nn.Linear(edge_dim, hidden_dim)
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for _ in range(num_layers):
            seq = nn.Sequential(nn.Linear(hidden_dim, hidden_dim * 2), nn.BatchNorm1d(hidden_dim * 2), nn.ReLU(), nn.Linear(hidden_dim * 2, hidden_dim))
            self.convs.append(GINEConv(seq))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
            
        self.gbn = nn.BatchNorm1d(global_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * 3 + global_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.BatchNorm1d(hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout), 
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward(self, x, ei, ea, batch, gf):
        x = self.node_emb(x)
        ea = self.edge_emb(ea)
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x, ei, ea) + x))
        xm = global_mean_pool(x, batch)
        xa = global_max_pool(x, batch)
        xs = global_add_pool(x, batch)
        return self.mlp(torch.cat([xm, xa, xs, self.gbn(gf)], 1))

def evaluate_gin_classifier(model, loader, device):
    model.eval()
    yt, yp_logits, yp_probs = [], [], []
    with torch.no_grad():
        for d in loader:
            d = d.to(device)
            out = model(d.x, d.edge_index, d.edge_attr, d.batch, d.global_feats)
            probs = torch.sigmoid(out)
            yt.extend(d.y.cpu().numpy().flatten())
            yp_probs.extend(probs.cpu().numpy().flatten())
            
    yt = np.array(yt)
    yp_probs = np.array(yp_probs)
    yp_preds = (yp_probs >= 0.5).astype(int)
    
    auc = roc_auc_score(yt, yp_probs) if len(np.unique(yt)) > 1 else 0.0
    pr_auc = average_precision_score(yt, yp_probs) if len(np.unique(yt)) > 1 else 0.0
    acc = accuracy_score(yt, yp_preds)
    
    return auc, pr_auc, acc

In [10]:
import torch
import torch.nn as nn
from pathlib import Path
from torch_geometric.loader import DataLoader as PyGDataLoader

DATA_PATH = Path(r"C:\Users\User\Desktop\spark_airflow\chembl\dags\data\gold\version_v2.0_20260606_1226")

train_ds = GraphQSARDatasetClassification(DATA_PATH / "scaffold_train.parquet", augment=True)
val_ds = GraphQSARDatasetClassification(DATA_PATH / "scaffold_val.parquet", augment=False)
test_ds = GraphQSARDatasetClassification(DATA_PATH / "scaffold_test.parquet", augment=False)

train_loader = PyGDataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)
val_loader = PyGDataLoader(val_ds, batch_size=256, shuffle=False, num_workers=0)
test_loader = PyGDataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)

s = next(iter(train_loader))
n_dim, e_dim, g_dim = s.num_node_features, s.edge_attr.shape[1], s.global_feats.shape[1]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GINAdvancedClassifier(
    node_dim=n_dim, edge_dim=e_dim, global_dim=g_dim,
    hidden_dim=128, num_layers=3, dropout=0.2
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
crit = nn.BCEWithLogitsLoss()

EPOCHS = 50
best_val_auc = 0.0

for ep in range(EPOCHS):
    model.train()
    total_loss = 0
    for d in train_loader:
        d = d.to(device)
        optimizer.zero_grad()
        out = model(d.x, d.edge_index, d.edge_attr, d.batch, d.global_feats)
        loss = crit(out, d.y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * d.num_graphs
        
    train_loss = total_loss / len(train_loader.dataset)
    val_auc, val_acc = evaluate_gin_classifier(model, val_loader, device)
    scheduler.step(val_auc)
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), "best_model_50ep.pt")
        
    print(f"Epoch {ep+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f}")

model.load_state_dict(torch.load("best_model_50ep.pt", weights_only=True))
test_auc, test_acc = evaluate_gin_classifier(model, test_loader, device)

print(f"\nFinal Test AUC (Best Model): {test_auc:.4f}")
print(f"Final Test Accuracy: {test_acc:.4f}")

ValueError: too many values to unpack (expected 2)